# EV Stock Harmonisation

Estimates the number of battery-electric passenger cars of vehicle class M1
per Austrian municipality at 2025-12-31.

Method: a hierarchical, mass-preserving proxy chain

```
federal-state BEV stock
    -> registration district, weighted by observed passenger-car stock
    -> municipality, weighted by population
```

This is **not** a statistically calibrated multi-proxy method. No wallbox,
charging-point, charging-access, charging-power, charging-profile,
building-allocation, or temporal-scenario logic is implemented here — this
workflow is exclusively concerned with harmonising municipal BEV stock.

The sole canonical vehicle-stock source is `kfz-bestand_2025.ods`
(`tab_1`, `tab_3`, `tab_6`, `tab_7`). `PKW_Bundesland_Energiequelle.csv` is a
redundant control extract and is not read here. `kfz_bestand_pro_1000.csv` is
a validation-only reference (different reference date, political rather than
registration-district level) and does not affect allocation weights.

The complete municipality-to-registration-district mapping is generated
reproducibly inside this notebook: deterministic default mapping, documented
single-municipality overrides, and a generic ambiguity detector for any
political district that corresponds to more than one registration district in
the source data. Ambiguous cases are written to an editable Excel review
workbook (`outputs/registration_district_review.xlsx`); final harmonised
outputs remain blocked until every ambiguous municipality has an approved,
valid assignment.


## 1. Install packages

In [ ]:
%pip install pandas openpyxl odfpy pyarrow

## 2. Imports and paths

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

from openpyxl.formatting.rule import FormulaRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter


CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not find the project data folder. "
        "Run the notebook from the project root or the notebooks folder."
    )

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Data folder:", DATA_DIR.resolve())
print("Output folder:", OUTPUT_DIR.resolve())


## 3. Configuration

In [ ]:
# Vehicle-stock and population reference dates are tracked separately.
# The official municipality population at 1 January 2026 is used as the
# spatial weighting variable. This date immediately follows the
# vehicle-stock reference date of 31 December 2025; it is not a replacement
# value for a missing 31 December population figure.
REFERENCE_DATE = pd.Timestamp("2025-12-31")
POPULATION_REFERENCE_DATE = pd.Timestamp("2026-01-01")

KFZ_BESTAND_PATH = DATA_DIR / "kfz-bestand_2025.ods"
POPULATION_PATH = DATA_DIR / "population_austria.ods"
MUNICIPALITIES_PATH = DATA_DIR / "municipalities_clean.xlsx"

REVIEW_WORKBOOK_PATH = OUTPUT_DIR / "registration_district_review.xlsx"
OUTPUT_XLSX_PATH = OUTPUT_DIR / "ev_municipality_harmonised.xlsx"
AUDIT_PARQUET_PATH = OUTPUT_DIR / "ev_municipality_audit.parquet"

GKZ_PREFIX_TO_STATE_CODE = {
    "1": "Burgenland",
    "2": "Kärnten",
    "3": "Niederösterreich",
    "4": "Oberösterreich",
    "5": "Salzburg",
    "6": "Steiermark",
    "7": "Tirol",
    "8": "Vorarlberg",
    "9": "Wien",
}

# Documented, single-municipality overrides (validated in Section 10a): the
# default political-district mapping is not sufficient for these municipalities
# because their registration district differs from the rest of their
# political district in the source table (tab_7). Format:
# municipality_id -> registration_district_name.
MUNICIPALITY_REGISTRATION_DISTRICT_OVERRIDES = {
    # Eisenstadt: tab_7 merges Eisenstadt's registration district with Rust
    # ("Eisenstadt (Stadt inkl. Rust)"), so the bare political-district name
    # "Eisenstadt (Stadt)" does not match the tab_7 label directly.
    "10101": "Eisenstadt (Stadt inkl. Rust)",
    # Rust: has no separate registration district in tab_7; vehicles
    # registered in Rust are recorded together with Eisenstadt.
    "10201": "Eisenstadt (Stadt inkl. Rust)",
    # Klosterneuburg: tab_7 lists a separate registration district
    # ("Klosterneuburg (BH Tulln)"), carved out of political district Tulln.
    "32144": "Klosterneuburg (BH Tulln)",
    # Schwechat: tab_7 lists a separate registration district ("Schwechat"),
    # carved out of political district Bruck an der Leitha.
    "30740": "Schwechat",
    # Leoben city: tab_7 splits political district Leoben into
    # "Leoben (Stadt)" and "Leoben (Land)"; the city itself uses "Stadt".
    "61108": "Leoben (Stadt)",
}

# Documented, district-wide override (validated in Section 10a): municipalities of a
# political district that are NOT covered by a single-municipality override
# above receive this registration district. Used only where the source
# table splits a political district into a city part (handled above) and a
# remaining "Land" part.
POLITICAL_DISTRICT_REMAINDER_OVERRIDES = {
    "611": "Leoben (Land)",
}

REQUIRED_REVIEW_COLUMNS = [
    "municipality_id",
    "municipality_name",
    "political_district_id",
    "political_district_name",
    "review_reason",
    "candidate_registration_districts",
    "suggested_registration_district",
    "review_status",
    "approved_registration_district",
    "review_note",
]

VALID_REVIEW_STATUS_VALUES = [
    "pending_review",
    "approved",
    "rejected",
    "insufficient_information",
]

CANDIDATE_SEPARATOR = " | "

REQUIRED_INPUT_PATHS = {
    "kfz-bestand_2025.ods (sole canonical vehicle-stock source)": KFZ_BESTAND_PATH,
    "population_austria.ods (municipality and political-district population)": POPULATION_PATH,
    "municipalities_clean.xlsx (municipality master data)": MUNICIPALITIES_PATH,
}
missing_required_inputs = [
    f"{label}: {path.resolve()}" for label, path in REQUIRED_INPUT_PATHS.items() if not path.exists()
]
if missing_required_inputs:
    raise FileNotFoundError(
        "The following required input file(s) are missing:\n"
        + "\n".join(f"  - {entry}" for entry in missing_required_inputs)
    )

print("All required input files were found.")
print("REFERENCE_DATE:", REFERENCE_DATE.date())
print("POPULATION_REFERENCE_DATE:", POPULATION_REFERENCE_DATE.date())


## 4. General helper functions

In [ ]:
def find_header_row(frame: pd.DataFrame, marker: str, search_rows: int = 15) -> int:
    """Find the row whose first column exactly equals `marker`."""
    for row_index in range(min(search_rows, len(frame))):
        if str(frame.iloc[row_index, 0]).strip() == marker:
            return row_index
    raise ValueError(f"Header marker '{marker}' not found.")


def normalize_district_name(name: str) -> str:
    """Deterministic, non-fuzzy normalisation for exact district-name comparison.

    Applies only: whitespace collapse, casefold, two documented fixed
    abbreviation expansions used in the Statistik Austria source tables
    (St. -> Sankt, Wr. -> Wiener), removal of parenthesis punctuation while
    keeping the enclosed qualifier word (e.g. "(Stadt)" -> " Stadt"), and
    removal of a trailing, purely generic "Bezirk" qualifier word. This is
    not fuzzy string matching: there is no similarity scoring and no
    tolerance for partial or approximate matches.
    """
    text = str(name).strip()
    text = re.sub(r"\bSt\.\s*", "Sankt ", text)
    text = re.sub(r"\bWr\.\s*", "Wiener ", text)
    text = text.replace("(", " ").replace(")", " ")
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+Bezirk$", "", text, flags=re.IGNORECASE)
    return text.casefold()


def safe_share(numerator, denominator):
    """Division that raises on a zero or missing denominator instead of
    silently producing inf/NaN."""
    if pd.isna(denominator) or denominator == 0:
        raise ValueError("Attempted to divide by a zero or missing denominator.")
    return numerator / denominator


def largest_remainder_apportionment(values: pd.Series, total: int) -> pd.Series:
    """Integer, sum-preserving allocation of `total` using the Largest
    Remainder method.

    `values` must be indexed by `municipality_id`. Tie-breaking is explicit
    and does not rely on the input order of `values` or on sort stability:
    winners are ranked by an explicit compound key of (fractional remainder
    descending, municipality_id ascending), built and sorted as its own
    step, so the result is identical regardless of the order municipalities
    appear in `values`.
    """
    floor_values = np.floor(values).astype(int)
    remainder = values - floor_values

    shortfall = int(total) - int(floor_values.sum())
    assert shortfall >= 0, (
        "Largest Remainder allocation failed: the sum of floor values "
        "exceeds the target total."
    )

    tie_break_key = pd.DataFrame(
        {
            "remainder": remainder.to_numpy(),
            "municipality_id": remainder.index.to_numpy(),
        }
    )
    # The source index is deliberately not reused as the DataFrame's index:
    # keeping "municipality_id" only as a plain column (with a default
    # RangeIndex) avoids any ambiguity between an index level and a column
    # of the same name when sorting.
    ranked_positions = tie_break_key.sort_values(
        ["remainder", "municipality_id"], ascending=[False, True], kind="mergesort"
    ).index
    ranked_index = remainder.index[ranked_positions]
    winners = ranked_index[:shortfall]

    result = floor_values.copy()
    result.loc[winners] += 1

    assert result.sum() == total
    return result


### 4a. Deterministic tie-break test

A small, artificial test verifying that the Largest Remainder tie-break is
determined solely by (fractional remainder descending, municipality_id
ascending), not by the order municipalities happen to appear in the input
`Series`. The same three tied values are tested in two different input
orderings; both must produce the identical result, and the smallest
`municipality_id` among the tied entries must always receive the
additional unit.

In [ ]:
def _run_tie_break_test():
    tied_value = 2.5
    municipality_ids = ["30003", "10001", "20002"]  # deliberately not sorted

    for shuffled_ids in (municipality_ids, list(reversed(municipality_ids))):
        test_values = pd.Series([tied_value] * 3, index=shuffled_ids)
        test_result = largest_remainder_apportionment(test_values, total=8)

        # floors = [2, 2, 2] (sum 6), shortfall = 2 -> two of the three tied
        # entries receive +1. With an explicit ascending-municipality_id
        # tie-break, the winners must be the two smallest IDs: 10001, 20002.
        expected = pd.Series({"10001": 3, "20002": 3, "30003": 2})
        actual = test_result.reindex(expected.index)
        assert actual.astype(int).equals(expected.astype(int)), (
            f"Tie-break test failed for input order {shuffled_ids}: "
            f"expected {expected.to_dict()}, got {actual.to_dict()}"
        )
        assert actual.sum() == 8

    print("Deterministic tie-break test passed: smallest municipality_id")
    print("among equal fractional remainders always receives the additional")
    print("unit, regardless of input order.")


_run_tie_break_test()


### 5. Review-workbook helper functions

In [ ]:
def read_review_table(review_path: Path) -> pd.DataFrame:
    """Read the manually editable Excel review workbook."""
    if not review_path.exists():
        return pd.DataFrame(columns=REQUIRED_REVIEW_COLUMNS)

    try:
        table = pd.read_excel(
            review_path,
            sheet_name="registration_district_review",
            dtype="string",
            engine="openpyxl",
        )
    except ValueError as error:
        raise ValueError(
            "The review workbook does not contain a worksheet named "
            "'registration_district_review'."
        ) from error

    missing_columns = [c for c in REQUIRED_REVIEW_COLUMNS if c not in table.columns]
    if missing_columns:
        raise ValueError(
            "Existing review workbook is missing columns: " + ", ".join(missing_columns)
        )
    return table


def merge_existing_review_decisions(
    generated_review: pd.DataFrame, review_path: Path
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Preserve manual decisions from an existing review workbook.

    Returns (merged_review, stale_rows). `stale_rows` lists rows from the
    existing workbook whose municipality no longer corresponds to a
    currently unresolved mapping (e.g. because the underlying source data
    changed); these rows are reported, not silently discarded from the report.
    """
    existing = read_review_table(review_path)

    if existing.empty:
        return generated_review, pd.DataFrame(columns=REQUIRED_REVIEW_COLUMNS)

    decision_columns = ["municipality_id", "review_status", "approved_registration_district", "review_note"]
    decisions = existing[decision_columns].drop_duplicates(subset="municipality_id", keep="last")

    merged = generated_review.merge(
        decisions, on="municipality_id", how="left", suffixes=("", "_existing")
    )
    for column in ["review_status", "approved_registration_district", "review_note"]:
        existing_column = f"{column}_existing"
        has_existing_value = merged[existing_column].notna()
        merged[column] = merged[column].mask(has_existing_value, merged[existing_column])
        merged = merged.drop(columns=existing_column)

    stale_rows = existing[
        ~existing["municipality_id"].isin(generated_review["municipality_id"])
    ].copy()

    return merged, stale_rows


def write_review_workbook(review_table: pd.DataFrame, review_path: Path) -> None:
    """Write a filterable, manually editable Excel review workbook.

    Formatting follows the PV harmonisation review-workbook pattern: frozen
    header row, autofilter, a formatted table, dropdown validation on
    `review_status`, per-candidate-group dropdown validation on
    `approved_registration_district`, status-based conditional formatting,
    highlighted editable columns, and an accompanying instructions sheet.
    """
    review_export = review_table[REQUIRED_REVIEW_COLUMNS].copy()

    instructions = pd.DataFrame(
        {
            "Field or step": [
                "Purpose",
                "First run",
                "review_status",
                "approved_registration_district",
                "review_note",
                "Approval rule",
                "Do not approve",
                "Rerun",
                "Important",
            ],
            "Instruction": [
                "This workbook lists municipalities whose political district "
                "corresponds to more than one registration district in the "
                "source data, together with the candidate registration "
                "districts found for that political district.",
                "No candidate is approved automatically. Review the rows in "
                "the 'registration_district_review' sheet.",
                "Set this field to 'approved' only after confirming the "
                "correct registration district for this municipality.",
                "Enter one of the candidate registration districts listed "
                "in 'candidate_registration_districts' for this row.",
                "Document the basis of the decision, for example: "
                "'Confirmed against an authoritative registration-district "
                "reference applicable to this data release'.",
                "Use the political district, the candidate list, and an "
                "authoritative administrative reference together.",
                "Do not approve rows with insufficient or contradictory "
                "evidence; use 'insufficient_information' or 'rejected' instead.",
                "Save and close this workbook, then rerun the notebook. "
                "Existing decisions are preserved.",
                "Do not change municipality_id, municipality_name, "
                "political_district_id, political_district_name, "
                "review_reason, candidate_registration_districts, or "
                "suggested_registration_district.",
            ],
        }
    )

    try:
        with pd.ExcelWriter(review_path, engine="openpyxl") as writer:
            review_export.to_excel(writer, sheet_name="registration_district_review", index=False)
            instructions.to_excel(writer, sheet_name="instructions", index=False)

            review_sheet = writer.sheets["registration_district_review"]
            instruction_sheet = writer.sheets["instructions"]

            review_sheet.freeze_panes = "A2"
            review_sheet.auto_filter.ref = review_sheet.dimensions
            review_sheet.sheet_view.showGridLines = False

            instruction_sheet.freeze_panes = "A2"
            instruction_sheet.sheet_view.showGridLines = False

            header_fill = PatternFill(fill_type="solid", fgColor="1F4E78")
            header_font = Font(bold=True, color="FFFFFF")
            for worksheet in [review_sheet, instruction_sheet]:
                for cell in worksheet[1]:
                    cell.fill = header_fill
                    cell.font = header_font
                    cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

            column_positions = {cell.value: cell.column for cell in review_sheet[1]}

            editable_fill = PatternFill(fill_type="solid", fgColor="FFF2CC")
            editable_columns = ["review_status", "approved_registration_district", "review_note"]
            for column_name in editable_columns:
                column_number = column_positions.get(column_name)
                if column_number is None:
                    continue
                for row_number in range(2, review_sheet.max_row + 1):
                    review_sheet.cell(row=row_number, column=column_number).fill = editable_fill

            text_columns = ["municipality_id", "political_district_id"]
            for column_name in text_columns:
                column_number = column_positions.get(column_name)
                if column_number is None:
                    continue
                for row_number in range(2, review_sheet.max_row + 1):
                    review_sheet.cell(row=row_number, column=column_number).number_format = "@"

            if review_sheet.max_row >= 2:
                status_column = column_positions.get("review_status")
                if status_column is not None:
                    status_letter = get_column_letter(status_column)
                    status_validation = DataValidation(
                        type="list",
                        formula1='"' + ",".join(VALID_REVIEW_STATUS_VALUES) + '"',
                        allow_blank=False,
                    )
                    status_validation.error = "Select one of the configured review-status values."
                    status_validation.errorTitle = "Invalid review status"
                    status_validation.prompt = "Set to 'approved' only after manual verification."
                    status_validation.promptTitle = "Review status"
                    review_sheet.add_data_validation(status_validation)
                    status_validation.add(f"{status_letter}2:{status_letter}{review_sheet.max_row}")

                    approved_rule = FormulaRule(
                        formula=[f'${status_letter}2="approved"'],
                        fill=PatternFill(fill_type="solid", fgColor="D9EAD3"),
                    )
                    rejected_rule = FormulaRule(
                        formula=[f'${status_letter}2="rejected"'],
                        fill=PatternFill(fill_type="solid", fgColor="F4CCCC"),
                    )
                    pending_rule = FormulaRule(
                        formula=[f'${status_letter}2="pending_review"'],
                        fill=PatternFill(fill_type="solid", fgColor="FFF9CC"),
                    )
                    insufficient_rule = FormulaRule(
                        formula=[f'${status_letter}2="insufficient_information"'],
                        fill=PatternFill(fill_type="solid", fgColor="FCE5CD"),
                    )
                    full_range = f"A2:{get_column_letter(review_sheet.max_column)}{review_sheet.max_row}"
                    review_sheet.conditional_formatting.add(full_range, approved_rule)
                    review_sheet.conditional_formatting.add(full_range, rejected_rule)
                    review_sheet.conditional_formatting.add(full_range, pending_rule)
                    review_sheet.conditional_formatting.add(full_range, insufficient_rule)

                approved_column = column_positions.get("approved_registration_district")
                if approved_column is not None:
                    approved_letter = get_column_letter(approved_column)
                    # One dropdown per distinct candidate group, restricted to
                    # exactly that group's own candidates (not the full union).
                    for candidates, group in review_export.groupby("candidate_registration_districts"):
                        options = [c.strip() for c in str(candidates).split(CANDIDATE_SEPARATOR) if c.strip()]
                        if not options:
                            continue
                        candidate_validation = DataValidation(
                            type="list",
                            formula1='"' + ",".join(options) + '"',
                            allow_blank=True,
                        )
                        candidate_validation.error = "Select one of the candidate registration districts for this row."
                        candidate_validation.errorTitle = "Invalid registration district"
                        review_sheet.add_data_validation(candidate_validation)
                        for row_position in group.index:
                            excel_row = row_position + 2
                            candidate_validation.add(f"{approved_letter}{excel_row}")

                review_table_object = Table(displayName="RegistrationDistrictReviewTable", ref=review_sheet.dimensions)
                review_table_object.tableStyleInfo = TableStyleInfo(
                    name="TableStyleMedium2", showFirstColumn=False, showLastColumn=False,
                    showRowStripes=True, showColumnStripes=False,
                )
                review_sheet.add_table(review_table_object)

            width_by_name = {
                "municipality_id": 16,
                "municipality_name": 28,
                "political_district_id": 18,
                "political_district_name": 26,
                "review_reason": 26,
                "candidate_registration_districts": 44,
                "suggested_registration_district": 30,
                "review_status": 30,
                "approved_registration_district": 30,
                "review_note": 46,
            }
            for column_number in range(1, review_sheet.max_column + 1):
                column_name = review_sheet.cell(row=1, column=column_number).value
                review_sheet.column_dimensions[get_column_letter(column_number)].width = width_by_name.get(column_name, 22)

            for row in review_sheet.iter_rows():
                for cell in row:
                    cell.alignment = Alignment(vertical="top", wrap_text=True)

            instruction_sheet.column_dimensions["A"].width = 30
            instruction_sheet.column_dimensions["B"].width = 95
            for row in instruction_sheet.iter_rows():
                for cell in row:
                    cell.alignment = Alignment(vertical="top", wrap_text=True)

    except PermissionError as error:
        raise PermissionError(
            f"Could not write the review workbook at {review_path}. "
            "Close the file in Excel (or any other program) and rerun this cell."
        ) from error


## 6. Load the municipality master table

In [ ]:
municipalities = pd.read_excel(MUNICIPALITIES_PATH, sheet_name="municipalities")

municipalities["municipality_id"] = municipalities["municipality_id"].astype(str).str.zfill(5)
municipalities["federal_state_code"] = municipalities["municipality_id"].str[0].map(GKZ_PREFIX_TO_STATE_CODE)
municipalities["political_district_id"] = municipalities["municipality_id"].str[:3]
municipalities["reference_date"] = REFERENCE_DATE

assert municipalities["municipality_id"].is_unique, "municipality_id is not unique."
assert municipalities["federal_state_code"].notna().all(), "Unmapped federal-state prefix found."
assert municipalities["municipality_name"].notna().all(), "Missing municipality name found."

check_municipality_count = len(municipalities) == 2092
check_vienna_present_once = (municipalities["municipality_id"] == "90001").sum() == 1

print("Municipalities:", len(municipalities))
print("Municipality count equals 2092:", check_municipality_count)
print("Vienna present exactly once:", check_vienna_present_once)
display(municipalities.head())


## 7. Load municipality population

In [ ]:
population_raw = pd.read_excel(POPULATION_PATH, sheet_name="Insgesamt", engine="odf", header=None)

header_row_index = find_header_row(population_raw, "Gebietseinheit")

population_date_column = None
for column_index in range(2, population_raw.shape[1]):
    header_value = population_raw.iloc[header_row_index, column_index]
    try:
        if pd.Timestamp(header_value) == POPULATION_REFERENCE_DATE:
            population_date_column = column_index
            break
    except (ValueError, TypeError):
        continue
assert population_date_column is not None, (
    f"Population column for {POPULATION_REFERENCE_DATE.date()} not found."
)

municipality_codes = population_raw[0].astype(str).str.strip()
is_municipality_row = municipality_codes.str.fullmatch(r"\d{5}")
municipality_population = population_raw.loc[is_municipality_row, [0, population_date_column]].copy()
municipality_population.columns = ["municipality_id", "population"]
municipality_population["municipality_id"] = municipality_population["municipality_id"].astype(str).str.strip()
municipality_population["population"] = pd.to_numeric(municipality_population["population"], errors="coerce")

# Political-district names (used only to label review rows and to build the
# default political-district-to-registration-district mapping; the same
# already-canonical population file, not a new input).
district_codes = population_raw[0].astype(str).str.strip()
is_district_row = district_codes.str.fullmatch(r"\d{3}")
political_districts = population_raw.loc[is_district_row, [0, 1]].copy()
political_districts.columns = ["political_district_id", "political_district_name"]
political_districts["political_district_id"] = political_districts["political_district_id"].astype(str).str.strip()
political_districts["political_district_name"] = political_districts["political_district_name"].astype(str).str.strip()
political_districts = political_districts[
    political_districts["political_district_id"].str[0].isin(GKZ_PREFIX_TO_STATE_CODE)
    & (political_districts["political_district_id"].str[0] != "9")
].reset_index(drop=True)
political_districts["federal_state_code"] = political_districts["political_district_id"].str[0].map(GKZ_PREFIX_TO_STATE_CODE)
political_districts["normalized_name"] = political_districts["political_district_name"].map(normalize_district_name)

municipality_ids_master = set(municipalities["municipality_id"])
municipality_ids_population = set(municipality_population["municipality_id"])
only_in_master = sorted(municipality_ids_master - municipality_ids_population)
only_in_population = sorted(municipality_ids_population - municipality_ids_master)

check_gkz_sets_equal = (not only_in_master) and (not only_in_population)
assert check_gkz_sets_equal, "Municipality ID sets differ between master data and population data."
assert municipality_population["population"].notna().all(), "Missing population value found."
assert (municipality_population["population"] > 0).all(), "Non-positive population value found."
assert not municipality_population.duplicated("municipality_id").any(), "Duplicate population record found."

municipalities = municipalities.merge(municipality_population, on="municipality_id", how="left", validate="one_to_one")
municipalities["population_reference_date"] = POPULATION_REFERENCE_DATE

print("Municipalities with a population value:", municipalities["population"].notna().sum())
print("Municipality ID sets equal (master vs. population):", check_gkz_sets_equal)
print("Political districts loaded:", len(political_districts))


## 8. Load federal-state EV stock

In [ ]:
tab3 = pd.read_excel(KFZ_BESTAND_PATH, sheet_name="tab_3", engine="odf", header=None)

tab3_header_row = tab3.iloc[1]
state_columns = {
    str(value).strip(): column_index
    for column_index, value in enumerate(tab3_header_row)
    if column_index >= 1
}

elektro_row = tab3[tab3[0].astype(str).str.strip() == "Elektro"].iloc[0]
pkw_total_row = tab3[tab3[0].astype(str).str.strip() == "Pkw insgesamt"].iloc[0]

FEDERAL_STATES = [s for s in state_columns if s != "Österreich"]

bev_by_state = {state: int(elektro_row[state_columns[state]]) for state in FEDERAL_STATES}
pkw_total_by_state = {state: int(pkw_total_row[state_columns[state]]) for state in FEDERAL_STATES}

tab1 = pd.read_excel(KFZ_BESTAND_PATH, sheet_name="tab_1", engine="odf", header=None)
national_pkw_m1 = int(tab1[tab1[0].astype(str).str.strip() == "Personenkraftwagen Kl. M1 insgesamt"].iloc[0][1])
national_bev = int(tab1[tab1[0].astype(str).str.strip() == "Elektro"].iloc[0][1])

check_tab3_matches_tab1_bev = int(elektro_row[state_columns["Österreich"]]) == national_bev
check_tab3_matches_tab1_pkw = int(pkw_total_row[state_columns["Österreich"]]) == national_pkw_m1
assert check_tab3_matches_tab1_bev, "tab_3 national BEV total does not match tab_1."
assert check_tab3_matches_tab1_pkw, "tab_3 national passenger-car total does not match tab_1."

print("BEV stock by federal state:", bev_by_state)
print("Passenger-car stock by federal state:", pkw_total_by_state)
print("National passenger-car (Kl. M1) total:", national_pkw_m1, " National BEV total:", national_bev)
print("tab_3 vs. tab_1 cross-check (BEV):", check_tab3_matches_tab1_bev)
print("tab_3 vs. tab_1 cross-check (passenger cars):", check_tab3_matches_tab1_pkw)


## 9. Load passenger-car stock by registration district

`tab_7` lists, per federal state, a "... insgesamt" header row followed by
the individual registration-district rows. Vienna's sub-rows (23 city
districts plus three non-geographic rows: police/judiciary/tax
administration, post, and rail) are skipped, since Vienna is handled
directly at federal-state level (Section 17).

In [ ]:
tab7 = pd.read_excel(KFZ_BESTAND_PATH, sheet_name="tab_7", engine="odf", header=None)

registration_district_records = []
current_state = None
skip_state_subrows = False

for _, row in tab7.iloc[2:].iterrows():
    label = str(row[0]).strip() if pd.notna(row[0]) else ""
    pkw_value = row[2]

    if label == "Österreich":
        continue
    if label.startswith("Q:"):
        break

    group_match = re.match(r"^(.+?) insgesamt$", label)
    if group_match and group_match.group(1) in FEDERAL_STATES:
        current_state = group_match.group(1)
        skip_state_subrows = current_state == "Wien"
        continue

    if skip_state_subrows or current_state is None or not label:
        continue

    registration_district_records.append(
        {
            "federal_state_code": current_state,
            "registration_district_name": label,
            "registration_district_pkw": pkw_value,
        }
    )

registration_districts = pd.DataFrame(registration_district_records)
registration_districts["normalized_name"] = registration_districts["registration_district_name"].map(
    normalize_district_name
)

assert not registration_districts.duplicated(["federal_state_code", "registration_district_name"]).any(), (
    "Duplicate registration-district rows found in tab_7."
)

pkw_sum_by_state = registration_districts.groupby("federal_state_code")["registration_district_pkw"].sum()
check_tab7_sums_match_tab3 = True
for state, summed_value in pkw_sum_by_state.items():
    if summed_value != pkw_total_by_state[state]:
        check_tab7_sums_match_tab3 = False
        print(f"MISMATCH {state}: sum of registration districts={summed_value} != tab_3 total={pkw_total_by_state[state]}")

assert check_tab7_sums_match_tab3, "Registration-district sums do not match tab_3 federal-state totals."

print("Registration-district rows (excluding Vienna):", len(registration_districts))
print("tab_7 sums match tab_3 federal-state totals:", check_tab7_sums_match_tab3)
display(registration_districts.head())


## 10. Build automatic registration-district mappings

Deterministic default mapping: a political district is assigned
automatically when its normalised name matches the normalised name of
exactly one registration district within the same federal state. Documented
single-municipality overrides (Section 3 configuration) are applied first
and take precedence over the default mapping.

Two explicit guards protect against silently picking an arbitrary match:
first, no two registration-district labels within the same federal state
may normalise to the same name (checked before the join); second, the join
result itself is checked so that no political district maps to more than
one registration-district row (checked before the mapping dictionary is
built). Either violation raises an explicit error rather than allowing
`dict` construction to silently overwrite a previous match.

In [ ]:
registration_name_collisions = registration_districts[
    registration_districts.duplicated(["federal_state_code", "normalized_name"], keep=False)
]
assert registration_name_collisions.empty, (
    "Two or more registration-district labels within the same federal state "
    "normalise to the same name; a direct match would be ambiguous rather "
    "than unique:\n"
    + registration_name_collisions[
        ["federal_state_code", "registration_district_name", "normalized_name"]
    ].to_string()
)

direct_match_candidates = political_districts.merge(
    registration_districts[["federal_state_code", "registration_district_name", "normalized_name"]],
    on=["federal_state_code", "normalized_name"],
    how="left",
)

match_counts_per_district = direct_match_candidates.groupby("political_district_id").size()
districts_with_multiple_direct_matches = match_counts_per_district[match_counts_per_district > 1]
assert districts_with_multiple_direct_matches.empty, (
    "The following political districts match more than one registration "
    "district by direct normalised-name comparison; automatic mapping must "
    "not silently pick the first match:\n"
    + districts_with_multiple_direct_matches.to_string()
)

direct_match_by_district = dict(
    zip(direct_match_candidates["political_district_id"], direct_match_candidates["registration_district_name"])
)

override_target_names = set(MUNICIPALITY_REGISTRATION_DISTRICT_OVERRIDES.values()) | set(
    POLITICAL_DISTRICT_REMAINDER_OVERRIDES.values()
)

resolved_count = sum(1 for v in direct_match_by_district.values() if pd.notna(v))
print("No within-state registration-district normalised-name collisions found.")
print("No political district matched more than one registration district directly.")
print("Political districts resolved by direct normalised-name match:", resolved_count)
print("Political districts not resolved by direct match:")
display(
    direct_match_candidates.loc[
        direct_match_candidates["registration_district_name"].isna(),
        ["political_district_id", "political_district_name", "federal_state_code"],
    ]
)


## 10a. Validate deterministic override rules

Every documented override (Section 3 configuration) listed explicitly,
together with the municipalities it actually affects in the current
municipality master data. This makes reported totals such as "5
single-municipality overrides + 15 district-remainder-override
municipalities" fully traceable from data, not asserted.

In [ ]:
override_validation_rows = []

for municipality_id, target in MUNICIPALITY_REGISTRATION_DISTRICT_OVERRIDES.items():
    municipality_row = municipalities.loc[municipalities["municipality_id"] == municipality_id]
    assert len(municipality_row) == 1, (
        f"Override municipality_id {municipality_id} not found in municipality master data."
    )
    override_validation_rows.append(
        {
            "override_type": "single_municipality",
            "political_district_id": municipality_row.iloc[0]["political_district_id"],
            "target_registration_district": target,
            "affected_municipality_ids": municipality_id,
            "number_of_affected_municipalities": 1,
        }
    )

for district_id, target in POLITICAL_DISTRICT_REMAINDER_OVERRIDES.items():
    remainder_mask = (municipalities["political_district_id"] == district_id) & (
        ~municipalities["municipality_id"].isin(MUNICIPALITY_REGISTRATION_DISTRICT_OVERRIDES)
    )
    remainder_municipality_ids = municipalities.loc[remainder_mask, "municipality_id"].tolist()
    assert remainder_municipality_ids, f"District-remainder override for {district_id} matches no municipality."
    override_validation_rows.append(
        {
            "override_type": "district_remainder",
            "political_district_id": district_id,
            "target_registration_district": target,
            "affected_municipality_ids": ", ".join(remainder_municipality_ids),
            "number_of_affected_municipalities": len(remainder_municipality_ids),
        }
    )

override_validation = pd.DataFrame(override_validation_rows)
display(
    override_validation[
        ["override_type", "political_district_id", "target_registration_district", "number_of_affected_municipalities"]
    ]
)

total_single_municipality_overrides = int(
    override_validation.loc[override_validation["override_type"] == "single_municipality", "number_of_affected_municipalities"].sum()
)
total_district_remainder_overrides = int(
    override_validation.loc[override_validation["override_type"] == "district_remainder", "number_of_affected_municipalities"].sum()
)
print(f"Single-municipality overrides: {total_single_municipality_overrides} municipalities")
print(f"District-remainder overrides: {total_district_remainder_overrides} municipalities")
print(
    "Total municipalities resolved by a documented override:",
    total_single_municipality_overrides + total_district_remainder_overrides,
)


## 11. Detect ambiguous registration-district mappings

Every candidate registration district associated with a political district
is recorded with the specific rule that found it and the textual evidence
for it — not merely the fact that some registration-district row was left
unused. Three rules apply, in this order:

1. **`direct_self_match`** — the political district's own normalised name
   equals a registration district's normalised name in the same federal
   state (Section 10). Recorded as a candidate for every political
   district that has one, regardless of whether that district later turns
   out to be ambiguous.
2. **`parenthetical_reference`** — an otherwise-unclaimed registration
   district row ends in "(&lt;name&gt;)", where `&lt;name&gt;` normalises to
   match another political district's own name in the same federal state.
3. **`unclaimed_fallback_unique_ambiguous_district_in_state`** — a
   registration-district row with no direct or referential attribution at
   all. This is only ever applied automatically when **exactly one**
   political district in the same federal state is already ambiguous from
   rules 1–2 (i.e. there is no other candidate district it could be
   confused with). If zero, or more than one, such district exists, this
   notebook raises an explicit error instead of guessing — this is what
   keeps the mechanism valid if a future data release contains more than
   one ambiguous political district, rather than silently relying on there
   being only one.

   Rule 3 is a **bounded candidate-generation fallback for manual review
   only**. It identifies that a registration-district row plausibly
   belongs to a political district's candidate set — nothing more. It
   does **not** prove which municipality within that district the row
   corresponds to, it does **not** replace an authoritative administrative
   reference applicable to the registration-district structure used in the
   vehicle-stock source, and it never approves or applies a final
   municipality assignment by itself: every municipality it produces a
   candidate list for still requires an explicit `approved` review
   decision (Section 13) before it can be included in the harmonised
   output.

A political district is generically ambiguous once it has **more than one**
associated candidate. This mechanism does not depend on the municipality
named Liezen, on the number 29, or on political district 612 as the only
possible case — see Section 11a for the full, auditable evidence trail on
the current data release.

In [ ]:
claimed_registration_district_names = set(override_target_names) | {
    v for v in direct_match_by_district.values() if pd.notna(v)
}

unclaimed_rows = registration_districts[
    ~registration_districts["registration_district_name"].isin(claimed_registration_district_names)
].copy()

district_name_lookup = political_districts.set_index("political_district_id")["political_district_name"].to_dict()
district_state_lookup = political_districts.set_index("political_district_id")["federal_state_code"].to_dict()

# Each record: which political district a candidate registration district
# was associated with, by which rule, and why.
candidate_evidence_records = []

# Rule 1: direct self-match.
for _, district in political_districts.iterrows():
    own_match = direct_match_by_district.get(district["political_district_id"])
    if pd.notna(own_match):
        candidate_evidence_records.append(
            {
                "political_district_id": district["political_district_id"],
                "candidate_registration_district": own_match,
                "candidate_detection_rule": "direct_self_match",
                "evidence": (
                    f"Registration district '{own_match}' normalises to "
                    f"'{normalize_district_name(own_match)}', matching political "
                    f"district name '{district['political_district_name']}'."
                ),
            }
        )

# Rule 2: parenthetical reference to another political district's own name.
# Zero candidates -> no reference found, leave to rule 3. Exactly one
# candidate -> apply. More than one candidate -> this is a genuine
# ambiguity in the reference itself and must not be resolved by picking
# the first match.
for _, unclaimed in unclaimed_rows.iterrows():
    state = unclaimed["federal_state_code"]
    raw_name = unclaimed["registration_district_name"]

    reference_match = re.search(r"\(([^)]+)\)\s*$", raw_name)
    if not reference_match:
        continue
    referenced_normalized = normalize_district_name(reference_match.group(1))
    state_districts = political_districts[political_districts["federal_state_code"] == state]

    exact_candidates = state_districts[state_districts["normalized_name"] == referenced_normalized]
    if len(exact_candidates) > 1:
        raise ValueError(
            f"Parenthetical reference in registration district '{raw_name}' "
            f"(federal state '{state}') matches more than one political district "
            f"by exact normalised name: {exact_candidates['political_district_id'].tolist()}. "
            "This is a genuine ambiguity and must not be resolved by picking the first match."
        )

    if len(exact_candidates) == 1:
        candidate_rows = exact_candidates
    else:
        # A reference may also be a suffix (e.g. "BH Tulln" referencing "Tulln").
        suffix_candidates = state_districts[
            state_districts["normalized_name"].map(lambda n: referenced_normalized.endswith(n))
        ]
        if len(suffix_candidates) > 1:
            raise ValueError(
                f"Parenthetical reference in registration district '{raw_name}' "
                f"(federal state '{state}') matches more than one political district "
                f"by suffix reference: {suffix_candidates['political_district_id'].tolist()}. "
                "This is a genuine ambiguity and must not be resolved by picking the first match."
            )
        if suffix_candidates.empty:
            continue
        candidate_rows = suffix_candidates

    referenced_district_id = candidate_rows.iloc[0]["political_district_id"]
    candidate_evidence_records.append(
        {
            "political_district_id": referenced_district_id,
            "candidate_registration_district": raw_name,
            "candidate_detection_rule": "parenthetical_reference",
            "evidence": (
                f"Registration district '{raw_name}' ends with a parenthetical "
                "reference matching political district "
                f"'{candidate_rows.iloc[0]['political_district_name']}'."
            ),
        }
    )

# Rule 3 (fallback), restricted to federal states with exactly one
# already-ambiguous political district from rules 1-2.
attributed_names = {r["candidate_registration_district"] for r in candidate_evidence_records}
still_unclaimed = unclaimed_rows[~unclaimed_rows["registration_district_name"].isin(attributed_names)]

district_candidate_counts = pd.Series(
    [r["political_district_id"] for r in candidate_evidence_records]
).value_counts()
districts_ambiguous_so_far = set(district_candidate_counts[district_candidate_counts > 1].index)

for _, unclaimed in still_unclaimed.iterrows():
    state = unclaimed["federal_state_code"]
    same_state_ambiguous = [
        district_id for district_id in districts_ambiguous_so_far if district_state_lookup.get(district_id) == state
    ]
    if len(same_state_ambiguous) != 1:
        raise ValueError(
            "Cannot generically attribute unclaimed registration-district row "
            f"'{unclaimed['registration_district_name']}' in federal state '{state}': "
            f"found {len(same_state_ambiguous)} already-ambiguous political district(s) "
            "there (a safe automatic attribution requires exactly 1). This row has no "
            "textual reference to any political district. Extend the detection rule "
            "explicitly before proceeding; do not resolve this by guessing."
        )
    district_id = same_state_ambiguous[0]
    candidate_evidence_records.append(
        {
            "political_district_id": district_id,
            "candidate_registration_district": unclaimed["registration_district_name"],
            "candidate_detection_rule": "unclaimed_fallback_unique_ambiguous_district_in_state",
            "evidence": (
                f"Registration district '{unclaimed['registration_district_name']}' has no "
                "direct or referential match to any political district and remains "
                f"unclaimed in federal state '{state}', where political district "
                f"'{district_name_lookup[district_id]}' ({district_id}) is the only "
                "already-ambiguous district found by rules 1-2; attributed to it as "
                "the documented, explicit fallback rule."
            ),
        }
    )

candidate_evidence = pd.DataFrame(candidate_evidence_records)

ambiguous_district_ids = [
    district_id
    for district_id, group in candidate_evidence.groupby("political_district_id")
    if group["candidate_registration_district"].nunique() > 1
]
ambiguous_district_candidates = {
    district_id: set(
        candidate_evidence.loc[
            candidate_evidence["political_district_id"] == district_id, "candidate_registration_district"
        ]
    )
    for district_id in ambiguous_district_ids
}

print("Generically detected ambiguous political districts:")
for district_id, candidates in ambiguous_district_candidates.items():
    print(f"  {district_id} ({district_name_lookup[district_id]}): {sorted(candidates)}")
if not ambiguous_district_candidates:
    print("  none")


## 11a. Candidate-detection diagnostics

Full, auditable evidence for how every candidate registration district was
associated with an ambiguous political district: which rule fired, and
why. This does not merely state that unused registration-district rows
remain — it shows the exact detection path for each one, for every
ambiguous political district found (not only the one currently present in
this data release).

In [ ]:
ambiguous_municipality_mask = municipalities["political_district_id"].isin(
    ambiguous_district_candidates
) & ~municipalities["municipality_id"].isin(MUNICIPALITY_REGISTRATION_DISTRICT_OVERRIDES)

affected_municipality_counts = municipalities.loc[ambiguous_municipality_mask].groupby("political_district_id").size()

candidate_detection_diagnostics = candidate_evidence[
    candidate_evidence["political_district_id"].isin(ambiguous_district_ids)
].copy()
candidate_detection_diagnostics["political_district_name"] = candidate_detection_diagnostics[
    "political_district_id"
].map(district_name_lookup)
candidate_detection_diagnostics["number_of_affected_municipalities"] = candidate_detection_diagnostics[
    "political_district_id"
].map(affected_municipality_counts)
candidate_detection_diagnostics = candidate_detection_diagnostics.rename(
    columns={"candidate_registration_district": "candidate_registration_districts"}
)[
    [
        "political_district_id",
        "political_district_name",
        "candidate_registration_districts",
        "candidate_detection_rule",
        "evidence",
        "number_of_affected_municipalities",
    ]
].sort_values(["political_district_id", "candidate_registration_districts"]).reset_index(drop=True)

display(candidate_detection_diagnostics)


## 12. Generate or update the editable Excel review workbook

One row per municipality belonging to a generically ambiguous political
district (and not already covered by a single-municipality override).

In [ ]:
# ambiguous_municipality_mask was already computed in Section 11a.
ambiguous_municipalities = municipalities.loc[ambiguous_municipality_mask].copy()

review_rows = []
for _, row in ambiguous_municipalities.iterrows():
    district_id = row["political_district_id"]
    candidates = sorted(ambiguous_district_candidates[district_id])
    review_rows.append(
        {
            "municipality_id": row["municipality_id"],
            "municipality_name": row["municipality_name"],
            "political_district_id": district_id,
            "political_district_name": district_name_lookup.get(district_id, pd.NA),
            "review_reason": "multiple_registration_districts",
            "candidate_registration_districts": CANDIDATE_SEPARATOR.join(candidates),
            # No principled ranking exists among the candidates without an
            # authoritative administrative reference applicable to the
            # registration-district structure used in the vehicle-stock
            # source; a suggestion would amount to a guess, so this is
            # intentionally left blank.
            "suggested_registration_district": pd.NA,
            "review_status": "pending_review",
            "approved_registration_district": pd.NA,
            "review_note": pd.NA,
        }
    )

generated_review = pd.DataFrame(review_rows, columns=REQUIRED_REVIEW_COLUMNS)

review_table, stale_review_rows = merge_existing_review_decisions(generated_review, REVIEW_WORKBOOK_PATH)
write_review_workbook(review_table, REVIEW_WORKBOOK_PATH)

print("Ambiguous municipalities requiring review:", len(review_table))
print("Review workbook:", REVIEW_WORKBOOK_PATH.resolve())
if not stale_review_rows.empty:
    print()
    print("Review rows no longer corresponding to a currently unresolved mapping:")
    display(stale_review_rows[["municipality_id", "municipality_name"]])

display(review_table.head(10))


## 13. Load approved review decisions

In [ ]:
current_review = read_review_table(REVIEW_WORKBOOK_PATH)

invalid_status_rows = current_review[
    ~current_review["review_status"].isin(VALID_REVIEW_STATUS_VALUES)
]
assert invalid_status_rows.empty, (
    "Review workbook contains rows with an invalid review_status value: "
    f"{invalid_status_rows['municipality_id'].tolist()}"
)

approved_rows = current_review[current_review["review_status"] == "approved"].copy()

def candidate_set(row):
    return {c.strip() for c in str(row["candidate_registration_districts"]).split(CANDIDATE_SEPARATOR) if c.strip()}

invalid_approved_rows = []
for _, row in approved_rows.iterrows():
    approved_value = row["approved_registration_district"]
    if pd.isna(approved_value) or str(approved_value).strip() == "":
        invalid_approved_rows.append(row["municipality_id"])
        continue
    if str(approved_value).strip() not in candidate_set(row):
        invalid_approved_rows.append(row["municipality_id"])

check_all_approved_rows_valid = len(invalid_approved_rows) == 0
print("Approved review rows:", len(approved_rows))
print("Approved rows with an empty or out-of-candidate-set assignment:", invalid_approved_rows)
print("All approved rows valid:", check_all_approved_rows_valid)


## 14. Apply reviewed registration-district assignments

In [ ]:
municipalities["registration_district_name"] = municipalities["political_district_id"].map(direct_match_by_district)
municipalities["mapping_method"] = "political_district_default"

for municipality_id, registration_district_name in MUNICIPALITY_REGISTRATION_DISTRICT_OVERRIDES.items():
    row_mask = municipalities["municipality_id"] == municipality_id
    municipalities.loc[row_mask, "registration_district_name"] = registration_district_name
    municipalities.loc[row_mask, "mapping_method"] = "documented_municipality_override"

for district_id, registration_district_name in POLITICAL_DISTRICT_REMAINDER_OVERRIDES.items():
    row_mask = (
        (municipalities["political_district_id"] == district_id)
        & (~municipalities["municipality_id"].isin(MUNICIPALITY_REGISTRATION_DISTRICT_OVERRIDES))
    )
    municipalities.loc[row_mask, "registration_district_name"] = registration_district_name
    municipalities.loc[row_mask, "mapping_method"] = "documented_district_remainder_override"

# Ambiguous municipalities: clear any tentative default, apply only an
# approved-and-valid review decision.
municipalities.loc[ambiguous_municipality_mask, "registration_district_name"] = pd.NA
municipalities.loc[ambiguous_municipality_mask, "mapping_method"] = "unresolved_pending_review"

approved_lookup = {
    row["municipality_id"]: str(row["approved_registration_district"]).strip()
    for _, row in approved_rows.iterrows()
    if row["municipality_id"] not in invalid_approved_rows
}
for municipality_id, registration_district_name in approved_lookup.items():
    row_mask = municipalities["municipality_id"] == municipality_id
    municipalities.loc[row_mask, "registration_district_name"] = registration_district_name
    municipalities.loc[row_mask, "mapping_method"] = "approved_review_decision"

municipalities["review_status"] = "not_applicable"
municipalities.loc[ambiguous_municipality_mask, "review_status"] = "pending_review"
for municipality_id in approved_lookup:
    municipalities.loc[municipalities["municipality_id"] == municipality_id, "review_status"] = "approved"

# Vienna: handled directly, no registration-district disaggregation.
vienna_mask = municipalities["federal_state_code"] == "Wien"
municipalities.loc[vienna_mask, "registration_district_name"] = pd.NA
municipalities.loc[vienna_mask, "mapping_method"] = "direct_state_to_municipality"
municipalities.loc[vienna_mask, "review_status"] = "not_applicable"

print(municipalities["mapping_method"].value_counts())
print()
print(municipalities["review_status"].value_counts())


## 15. Validate the complete mapping

In [ ]:
unresolved_municipalities = municipalities[
    (municipalities["federal_state_code"] != "Wien")
    & (
        municipalities["registration_district_name"].isna()
        | (municipalities["registration_district_name"].astype(str).str.strip() == "")
    )
].copy()

check_no_unresolved_mappings = len(unresolved_municipalities) == 0

non_vienna = municipalities[municipalities["federal_state_code"] != "Wien"]
check_every_municipality_one_mapping = (
    non_vienna["registration_district_name"].notna().all() if check_no_unresolved_mappings else False
)
check_no_duplicate_final_mapping = not municipalities.duplicated("municipality_id").any()

valid_registration_district_names = set(registration_districts["registration_district_name"]) | set(
    override_target_names
)
applied_names = non_vienna["registration_district_name"].dropna().unique().tolist()
names_not_in_source = [n for n in applied_names if n not in valid_registration_district_names]
check_applied_names_exist_in_source = len(names_not_in_source) == 0

blocked_states = sorted(unresolved_municipalities["federal_state_code"].unique().tolist())

print("Unresolved municipalities:", len(unresolved_municipalities))
print("Affected political district IDs:", sorted(unresolved_municipalities["political_district_id"].unique().tolist()))
print("Affected political district names:", sorted(unresolved_municipalities["political_district_name"].dropna().unique().tolist()) if "political_district_name" in unresolved_municipalities else "n/a")
print("Affected federal states (blocked):", blocked_states)
print()
print("Every non-Vienna municipality has exactly one mapping:", check_every_municipality_one_mapping)
print("No duplicate final mapping:", check_no_duplicate_final_mapping)
print("Every applied registration-district name exists in the tab_7 source:", check_applied_names_exist_in_source)

if not unresolved_municipalities.empty:
    display(
        unresolved_municipalities[
            ["municipality_id", "municipality_name", "political_district_id", "federal_state_code"]
        ]
    )


## 16. Derive municipality passenger-car proxies

For municipality *m* in registration district *z*:

```
A_m = C_z * (P_m / P_z)
```

where `C_z` is the observed passenger-car stock of registration district
*z*, `P_m` is the population of municipality *m*, and `P_z` is the
population sum of registration district *z*. Only resolved (non-blocked)
municipalities are processed.

In [ ]:
resolved_municipalities = municipalities[
    (municipalities["federal_state_code"] != "Wien")
    & municipalities["registration_district_name"].notna()
].copy()

resolved_municipalities = resolved_municipalities.merge(
    registration_districts[["registration_district_name", "registration_district_pkw"]],
    on="registration_district_name",
    how="left",
    validate="many_to_one",
)
assert resolved_municipalities["registration_district_pkw"].notna().all(), (
    "At least one applied registration-district name has no passenger-car "
    "stock value in tab_7."
)

resolved_municipalities["registration_district_population"] = resolved_municipalities.groupby(
    "registration_district_name"
)["population"].transform("sum")

resolved_municipalities["within_district_population_share"] = resolved_municipalities.apply(
    lambda r: safe_share(r["population"], r["registration_district_population"]), axis=1
)
resolved_municipalities["estimated_total_pkw_proxy"] = (
    resolved_municipalities["registration_district_pkw"]
    * resolved_municipalities["within_district_population_share"]
)

passenger_car_count_parts = []

for registration_district_name, district_data in (
    resolved_municipalities.groupby(
        "registration_district_name"
    )
):
    district_data = district_data.set_index(
        "municipality_id"
    )

    district_total = int(
        district_data[
            "registration_district_pkw"
        ].iloc[0]
    )

    counts = largest_remainder_apportionment(
        district_data[
            "estimated_total_pkw_proxy"
        ],
        total=district_total,
    )

    passenger_car_count_parts.append(
        counts.rename(
            "passenger_car_count"
        )
    )

passenger_car_counts = pd.concat(
    passenger_car_count_parts
)

resolved_municipalities[
    "passenger_car_count"
] = (
    resolved_municipalities[
        "municipality_id"
    ]
    .map(passenger_car_counts)
    .astype(int)
)

proxy_mass_check = resolved_municipalities.groupby("registration_district_name").agg(
    proxy_sum=("estimated_total_pkw_proxy", "sum"),
    official_value=("registration_district_pkw", "first"),
)
proxy_mass_check["abs_difference"] = (proxy_mass_check["proxy_sum"] - proxy_mass_check["official_value"]).abs()
check_pkw_proxy_mass_conserved = bool((proxy_mass_check["abs_difference"] < 1e-6).all())

print("Passenger-car proxy mass-conserved per registration district (tolerance 1e-6):", check_pkw_proxy_mass_conserved)
assert check_pkw_proxy_mass_conserved


## 17. Disaggregate EV stock

For municipality *m* in federal state *s*:

```
E_raw_m = E_s * (A_m / sum_j A_j)   for j in s
```

Equivalently, in hierarchical closed form:

```
E_raw_m = E_s * (C_z / C_s) * (P_m / P_z)
```

Vienna is handled directly (it is simultaneously a federal state and a
municipality): `unit_count = E_Wien`, no disaggregation.

In [ ]:
pkw_proxy_sum_by_state = resolved_municipalities.groupby("federal_state_code")["estimated_total_pkw_proxy"].sum()

fully_resolved_states = [
    state
    for state in FEDERAL_STATES
    if state != "Wien"
    and (municipalities.loc[municipalities["federal_state_code"] == state, "registration_district_name"].isna().sum() == 0)
]
blocked_states = [state for state in FEDERAL_STATES if state != "Wien" and state not in fully_resolved_states]

resolved_municipalities["district_pkw_share_of_state"] = resolved_municipalities.apply(
    lambda r: safe_share(r["registration_district_pkw"], pkw_total_by_state[r["federal_state_code"]]), axis=1
)
resolved_municipalities["state_total_pkw"] = resolved_municipalities["federal_state_code"].map(pkw_total_by_state)
resolved_municipalities["state_bev_source_count"] = resolved_municipalities["federal_state_code"].map(bev_by_state)

resolved_municipalities["allocation_weight"] = np.nan
resolved_municipalities["estimated_unit_count_raw"] = np.nan
for state in fully_resolved_states:
    state_mask = resolved_municipalities["federal_state_code"] == state
    resolved_municipalities.loc[state_mask, "allocation_weight"] = (
        resolved_municipalities.loc[state_mask, "estimated_total_pkw_proxy"] / pkw_proxy_sum_by_state[state]
    )
    resolved_municipalities.loc[state_mask, "estimated_unit_count_raw"] = (
        bev_by_state[state] * resolved_municipalities.loc[state_mask, "allocation_weight"]
    )

print("Fully resolved federal states (computable):", fully_resolved_states)
print("Blocked federal states (unresolved municipalities present):", blocked_states)

wien_row = municipalities[municipalities["federal_state_code"] == "Wien"].iloc[0]
wien_result = pd.DataFrame(
    [
        {
            "municipality_id": wien_row["municipality_id"],
            "municipality_name": wien_row["municipality_name"],
            "federal_state_code": "Wien",
            "reference_date": REFERENCE_DATE,
            "unit_count": bev_by_state["Wien"],
            "passenger_car_count": pkw_total_by_state["Wien"],
            "derivation_method": "direct_state_to_municipality",
            "data_quality": "reported",
        }
    ]
)
display(wien_result)


## 18. Integer allocation

Largest Remainder method, applied separately within each federal state, so
that the sum of integer `unit_count` values equals the observed federal-state
BEV stock exactly.

In [ ]:
final_state_frames = []
audit_state_frames = []

for state in fully_resolved_states:
    state_data = resolved_municipalities.loc[resolved_municipalities["federal_state_code"] == state].set_index(
        "municipality_id"
    )

    unit_count = largest_remainder_apportionment(state_data["estimated_unit_count_raw"], bev_by_state[state])
    floor_values = np.floor(state_data["estimated_unit_count_raw"]).astype(int)
    fractional_remainder = state_data["estimated_unit_count_raw"] - floor_values
    rounding_adjustment = unit_count > floor_values

    state_result = pd.DataFrame(
        {
            "municipality_id": state_data.index,
            "municipality_name": state_data["municipality_name"],
            "federal_state_code": state,
            "reference_date": REFERENCE_DATE,
            "unit_count": unit_count,
            "passenger_car_count": state_data["passenger_car_count"],
            "derivation_method": "registration_district_pkw_population_proxy",
            "data_quality": "estimated",
        }
    ).reset_index(drop=True)
    final_state_frames.append(state_result)

    audit_columns = state_data.reset_index()[
        [
            "municipality_id",
            "municipality_name",
            "federal_state_code",
            "political_district_id",
            "registration_district_name",
            "mapping_method",
            "review_status",
            "population",
            "population_reference_date",
            "registration_district_population",
            "registration_district_pkw",
            "state_total_pkw",
            "state_bev_source_count",
            "within_district_population_share",
            "district_pkw_share_of_state",
            "estimated_total_pkw_proxy",
            "allocation_weight",
            "estimated_unit_count_raw",
        ]
    ].copy()
    audit_columns["political_district_name"] = audit_columns["political_district_id"].map(district_name_lookup)
    audit_columns["review_note"] = pd.NA
    audit_columns["estimated_unit_count_floor"] = floor_values.values
    audit_columns["fractional_remainder"] = fractional_remainder.values
    audit_columns["rounding_adjustment"] = rounding_adjustment.values
    audit_columns["unit_count"] = unit_count.values
    audit_columns["reference_date"] = REFERENCE_DATE
    audit_columns["derivation_method"] = "registration_district_pkw_population_proxy"
    audit_columns["data_quality"] = "estimated"
    audit_state_frames.append(audit_columns)

final_resolved = pd.concat(final_state_frames, ignore_index=True) if final_state_frames else pd.DataFrame()
audit_resolved = pd.concat(audit_state_frames, ignore_index=True) if audit_state_frames else pd.DataFrame()

print("Computed municipalities (excluding Vienna, excluding blocked federal states):", len(final_resolved))


## 19. Validation

In [ ]:
validation_rows = []


def add_check(name, passed, expected=None, actual=None):
    validation_rows.append({"check": name, "expected": expected, "actual": actual, "passed": bool(passed)})


add_check("municipality_count_2092", check_municipality_count, 2092, len(municipalities))
add_check("municipality_id_unique", municipalities["municipality_id"].is_unique, True, municipalities["municipality_id"].is_unique)
add_check("vienna_present_exactly_once", check_vienna_present_once, True, check_vienna_present_once)
add_check("gkz_sets_equal_master_vs_population", check_gkz_sets_equal, True, check_gkz_sets_equal)
add_check("tab3_matches_tab1_bev", check_tab3_matches_tab1_bev, True, check_tab3_matches_tab1_bev)
add_check("tab3_matches_tab1_pkw", check_tab3_matches_tab1_pkw, True, check_tab3_matches_tab1_pkw)
add_check("tab7_sums_match_tab3", check_tab7_sums_match_tab3, True, check_tab7_sums_match_tab3)
add_check("every_municipality_one_mapping", check_every_municipality_one_mapping, True, check_every_municipality_one_mapping)
add_check("no_duplicate_final_mapping", check_no_duplicate_final_mapping, True, check_no_duplicate_final_mapping)
add_check("applied_names_exist_in_tab7_source", check_applied_names_exist_in_source, True, check_applied_names_exist_in_source)
add_check("all_approved_review_rows_valid", check_all_approved_rows_valid, True, check_all_approved_rows_valid)
add_check("no_unresolved_review_cases", check_no_unresolved_mappings, True, check_no_unresolved_mappings)
add_check("no_blocked_federal_states", len(blocked_states) == 0, "[]", str(blocked_states))
add_check("pkw_proxy_mass_conserved_per_district", check_pkw_proxy_mass_conserved, True, check_pkw_proxy_mass_conserved)

for state in fully_resolved_states:
    state_sum = int(final_resolved.loc[final_resolved["federal_state_code"] == state, "unit_count"].sum())
    add_check(f"bev_mass_conserved_{state}", state_sum == bev_by_state[state], bev_by_state[state], state_sum)

add_check(
    "bev_mass_conserved_Wien",
    int(wien_result["unit_count"].sum()) == bev_by_state["Wien"],
    bev_by_state["Wien"],
    int(wien_result["unit_count"].sum()),
)

if not final_resolved.empty:
    combined_preview = pd.concat([final_resolved, wien_result], ignore_index=True)
    add_check("unit_count_is_integer", combined_preview["unit_count"].map(float).map(float.is_integer).all(), True, True)
    add_check("no_negative_unit_count", bool((combined_preview["unit_count"] >= 0).all()), True, True)
    add_check("no_missing_unit_count", bool(combined_preview["unit_count"].notna().all()), True, True)
    add_check("municipality_id_unique_in_output", not combined_preview["municipality_id"].duplicated().any(), True, True)
    if len(blocked_states) == 0:
        add_check(
            "national_bev_total_matches_257717",
            int(combined_preview["unit_count"].sum()) == 257717,
            257717,
            int(combined_preview["unit_count"].sum()),
        )

validation_report = pd.DataFrame(validation_rows)
display(validation_report)

all_critical_checks_passed = bool(validation_report["passed"].all())
print()
print("ALL CRITICAL CHECKS PASSED:", all_critical_checks_passed)
if not all_critical_checks_passed:
    print("Failed checks:", validation_report.loc[~validation_report["passed"], "check"].tolist())


## 20. Export harmonised outputs (blocked while critical checks fail)

No final harmonised outputs are written while `all_critical_checks_passed`
is not `True` — in particular, while any ambiguous municipality has no
review row, a `review_status` other than `approved`, an empty approved
registration district, or an approved registration district outside its
candidate set. The review workbook itself may still have been generated or
updated in Section 12 even when the final export is blocked.

In [ ]:
if all_critical_checks_passed:
    canonical_columns = [
        "municipality_id",
        "municipality_name",
        "federal_state_code",
        "reference_date",
        "unit_count",
        "passenger_car_count",
        "derivation_method",
        "data_quality",
    ]
    final_output = pd.concat([final_resolved, wien_result], ignore_index=True)[canonical_columns]
    final_output = final_output.sort_values("municipality_id").reset_index(drop=True)

    try:
        with pd.ExcelWriter(OUTPUT_XLSX_PATH, engine="openpyxl") as writer:
            final_output.to_excel(writer, sheet_name="ev_by_municipality", index=False)
            validation_report.to_excel(writer, sheet_name="validation", index=False)

            state_summary_rows = []
            for state in fully_resolved_states + ["Wien"]:
                computed_sum = int(
                    pd.concat([final_resolved, wien_result])
                    .loc[lambda d: d["federal_state_code"] == state, "unit_count"]
                    .sum()
                )
                state_summary_rows.append(
                    {"federal_state_code": state, "official_bev_stock": bev_by_state[state], "computed_unit_count_sum": computed_sum}
                )
            pd.DataFrame(state_summary_rows).to_excel(writer, sheet_name="federal_state_validation", index=False)

            registration_districts.drop(columns="normalized_name").to_excel(
                writer, sheet_name="registration_district_validation", index=False
            )

            final_mapping = municipalities[
                ["municipality_id", "municipality_name", "federal_state_code", "political_district_id",
                 "registration_district_name", "mapping_method", "review_status"]
            ].copy()
            final_mapping.to_excel(writer, sheet_name="final_registration_district_mapping", index=False)

            special_cases = pd.DataFrame(
                [
                    {"municipality_id": mid, "resolution": "documented_municipality_override", "registration_district_name": name}
                    for mid, name in MUNICIPALITY_REGISTRATION_DISTRICT_OVERRIDES.items()
                ]
                + [
                    {"municipality_id": "(remaining municipalities)", "resolution": f"documented_district_remainder_override (district {d})", "registration_district_name": name}
                    for d, name in POLITICAL_DISTRICT_REMAINDER_OVERRIDES.items()
                ]
                + [
                    {"municipality_id": "(Vienna, single municipality)", "resolution": "direct_state_to_municipality", "registration_district_name": "not applicable"}
                ]
            )
            special_cases.to_excel(writer, sheet_name="special_cases", index=False)

            override_validation.to_excel(writer, sheet_name="override_validation", index=False)
            candidate_detection_diagnostics.to_excel(writer, sheet_name="candidate_detection_diagnostics", index=False)

            source_metadata = pd.DataFrame(
                {
                    "source_file": [
                        "kfz-bestand_2025.ods",
                        "kfz-bestand_2025.ods",
                        "kfz-bestand_2025.ods",
                        "population_austria.ods",
                        "municipalities_clean.xlsx",
                        "kfz_bestand_pro_1000.csv",
                    ],
                    "sheet_or_table": ["tab_3 / tab_1", "tab_7", "tab_6", "Insgesamt", "municipalities", "(whole file)"],
                    "role": [
                        "Federal-state BEV stock and passenger-car stock; national control totals",
                        "Passenger-car stock by registration district",
                        "Canonical validation-only input: supplementary population plausibility check (not an allocation input, see validation_only_plausibility)",
                        "Municipality and political-district population",
                        "Municipality master table and identifiers",
                        "Canonical validation-only input: cross-period passenger-car plausibility check (not an allocation input, see validation_only_plausibility)",
                    ],
                }
            )
            source_metadata.to_excel(writer, sheet_name="source_metadata", index=False)

            municipality_id_column = writer.sheets["ev_by_municipality"]["A"]
            for cell in municipality_id_column[1:]:
                cell.number_format = "@"

    except PermissionError as error:
        raise PermissionError(
            f"Could not write {OUTPUT_XLSX_PATH}. Close the file in Excel (or any other "
            "program) and rerun this cell."
        ) from error

    audit_output = pd.concat(
        [
            audit_resolved,
            pd.DataFrame(
                [
                    {
                        "municipality_id": wien_row["municipality_id"],
                        "municipality_name": wien_row["municipality_name"],
                        "federal_state_code": "Wien",
                        "political_district_id": wien_row["political_district_id"],
                        "political_district_name": pd.NA,
                        "registration_district_name": pd.NA,
                        "mapping_method": "direct_state_to_municipality",
                        "review_status": "not_applicable",
                        "population": wien_row["population"],
                        "population_reference_date": POPULATION_REFERENCE_DATE,
                        "registration_district_population": pd.NA,
                        "registration_district_pkw": pd.NA,
                        "state_total_pkw": pkw_total_by_state["Wien"],
                        "state_bev_source_count": bev_by_state["Wien"],
                        "within_district_population_share": pd.NA,
                        "district_pkw_share_of_state": pd.NA,
                        "estimated_total_pkw_proxy": pd.NA,
                        "allocation_weight": pd.NA,
                        "estimated_unit_count_raw": float(bev_by_state["Wien"]),
                        "estimated_unit_count_floor": bev_by_state["Wien"],
                        "fractional_remainder": 0.0,
                        "rounding_adjustment": False,
                        "unit_count": bev_by_state["Wien"],
                        "reference_date": REFERENCE_DATE,
                        "review_note": pd.NA,
                        "derivation_method": "direct_state_to_municipality",
                        "data_quality": "reported",
                    }
                ]
            ),
        ],
        ignore_index=True,
    )
    audit_output.to_parquet(AUDIT_PARQUET_PATH, index=False)

    print("Created:", OUTPUT_XLSX_PATH.resolve())
    print("Created:", AUDIT_PARQUET_PATH.resolve())
else:
    print("FINAL EXPORT BLOCKED.")
    print("Reason: at least one critical check failed, and/or unresolved ambiguous")
    print("municipalities remain.")
    print()
    print("Unresolved municipalities:", len(unresolved_municipalities))
    print("Affected political district IDs:", sorted(unresolved_municipalities["political_district_id"].unique().tolist()))
    print(
        "Affected political district names:",
        sorted(political_districts.set_index("political_district_id")["political_district_name"]
               .reindex(unresolved_municipalities["political_district_id"].unique()).dropna().unique().tolist()),
    )
    print("Affected federal states:", blocked_states)
    print("Review workbook:", REVIEW_WORKBOOK_PATH.resolve())
    print()
    print("Next required action: open the review workbook above, set 'review_status' to")
    print("'approved' and 'approved_registration_district' to one of the listed candidates")
    print("for each unresolved municipality, save, close, and rerun this notebook.")
    print()
    print("No harmonised workbook or audit Parquet file was written.")


## 21. Final summary

In [ ]:
print("Municipalities total:", len(municipalities))
print("Approved:", (municipalities["review_status"] == "approved").sum())
print("Pending review:", (municipalities["review_status"] == "pending_review").sum())
print("Not applicable (Vienna / non-ambiguous default or override):", (municipalities["review_status"] == "not_applicable").sum())
print()
print("Fully computable federal states:", fully_resolved_states + ["Wien"])
print("Blocked federal states:", blocked_states)
print()
print("Final harmonised export completed:", all_critical_checks_passed)
